Libraries and Dataset

In [1]:
!pip install -q torch_geometric

import numpy as np
import pandas as pd
from torch_geometric.datasets import EllipticBitcoinDataset

dataset = EllipticBitcoinDataset(root="/content/elliptic")
data = dataset[0]
print(data)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.2 MB/s eta 0:00:00


Processing...


Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769])


Done!


Prepare Train and Test Sets

In [2]:
X = data.x.numpy()
y = data.y.numpy()
train_mask = data.train_mask.numpy()
test_mask = data.test_mask.numpy()

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("train:", X_train.shape, "| fraud:", y_train.sum(), f"({y_train.mean():.1%})")
print("test: ", X_test.shape, "| fraud:", y_test.sum(), f"({y_test.mean():.1%})")
print("unique labels:", np.unique(y_train), np.unique(y_test))

train: (29894, 165) | fraud: 3462 (11.6%)
test:  (16670, 165) | fraud: 1083 (6.5%)
unique labels: [0 1] [0 1]


Training and Evaluation Function

In [3]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score
)

# Weight of the minority class: number of legitimate transactions per fraud transaction
spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(spw, 2))


def run_xgb(cols, seed):
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(X_train[:, cols], y_train)

    proba = model.predict_proba(X_test[:, cols])[:, 1]  # Fraud probability
    pred = (proba >= 0.5).astype(int)                   # Threshold of 0.5

    return {
        "pr_auc": average_precision_score(y_test, proba),
        "roc_auc": roc_auc_score(y_test, proba),
        "f1_illicit": f1_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
    }

scale_pos_weight: 7.63


Run Experiments with 5 Random Seeds per Variant

In [6]:
variants = {
    "local (93 features, no graph)": np.arange(0, 93),
    "all (165 features, with aggregates)": np.arange(0, 165),
}

rows = []

for name, cols in variants.items():
    for seed in range(5):
        res = run_xgb(cols, seed)
        res.update({"model": name, "seed": seed})
        rows.append(res)

results = pd.DataFrame(rows)

metrics = [
    "pr_auc",
    "roc_auc",
    "f1_illicit",
    "f1_macro",
    "precision",
    "recall",
]

summary = results.groupby("model")[metrics].agg(["mean", "std"]).round(3)

summary

pr_auc        roc_auc        f1_illicit  \
                                      mean    std    mean    std       mean   
model                                                                         
all (165 features, with aggregates)  0.802  0.002   0.930  0.003      0.793   
local (93 features, no graph)        0.783  0.001   0.917  0.003      0.743   

                                           f1_macro        precision         \
                                       std     mean    std      mean    std   
model                                                                         
all (165 features, with aggregates)  0.006    0.890  0.003     0.859  0.014   
local (93 features, no graph)        0.004    0.863  0.002     0.765  0.007   

                                    recall         
                                      mean    std  
model                                              
all (165 features, with aggregates)  0.736  0.001  
local (93 features, no graph)        0.721  0.001

Save Experimental Results

In [5]:
import os

os.makedirs("results", exist_ok=True)

results.to_csv(
    "results/elliptic_baseline_xgboost.csv",
    index=False
)

summary.to_csv(
    "results/elliptic_baseline_xgboost_summary.csv"
)